In [ ]:
# COLORS       = {'Eastern Arabian Sea':'blood',
#                 'Central India':'#D42028',
#                 'Central Bay of Bengal':'#F2C85E',
#                 'Equatorial Indian Ocean':'#5BA7DA',
#                 'Konkan Coast':'#1B2C61'}

In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import sympy as sp
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100,'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SRCONFIG   = MODELS['sr']
SPLIT      = 'valid'
SEEDS      = SRCONFIG['seeds']

# The 8 target models in display order
MODELS_8 = ['pod_bl','base_bl','base_all','nonparam_all','gauss_all','sr_rh','sr_thermo','sr_thermo_flux']
LABELS_8 = {
    'pod_bl':        'POD',
    'base_bl':       r'Baseline NN ($B_L$)',
    'base_all':      'Baseline NN (all)',
    'nonparam_all':  'Nonparam. Kernel NN',
    'gauss_all':     'Param. Kernel NN',
    'sr_rh':         'SR-RH',
    'sr_thermo':     'SR-Thermo',
    'sr_thermo_flux':'SR-Thermo+Flux',
}
COLORS_8 = {
    'pod_bl':        'blood',
    'base_bl':       '#5BA7DA',
    'base_all':      '#5BA7DA',
    'nonparam_all':  '#F2C85E',
    'gauss_all':     '#D42028',
    'sr_rh':         '#1B2C61',
    'sr_thermo':     '#1B2C61',
    'sr_thermo_flux':'#1B2C61',
}

In [ ]:
def get_r2(ytrue,ypred,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    ssres = ((ytrue-ypred)**2).sum(dim=dims,skipna=True)
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    return float(1-ssres/sstot)

def get_mse(ytrue,ypred,dims=None):
    dims = list(ytrue.dims) if dims is None else dims
    return float(((ytrue-ypred)**2).mean(dim=dims,skipna=True))

def get_nn_complexity(kind,nfieldvars,nlevs,nlocalvars):
    def nparams(nfeatures):
        return (nfeatures*256)+256+(256*128)+128+(128*64)+64+(64*32)+32+(32*1)+1
    if kind=='baseline':
        return nparams(nfieldvars*nlevs+nlocalvars)
    elif kind=='nonparametric':
        return nfieldvars*nlevs+nparams(nfieldvars+nlocalvars)
    elif kind=='parametric':
        return 2*nfieldvars+nparams(nfieldvars+nlocalvars)

def get_mse_at_r2(ytrue,r2_target=0.5,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    n     = ytrue.count(dim=dims)
    return float((1-r2_target)*sstot/n)

def pareto_front(records):
    ordered,front,best = sorted(records,key=lambda r: r['mse']),[],np.inf
    for r in ordered:
        if r['nparams'] < best:
            front.append(r)
            best = r['nparams']
    return sorted(front,key=lambda r: r['nparams'])

def prettify_eq(eq_str):
    all_vars = ['rh','thetae','thetaestar','lf','shf','lhf']
    var_syms = {name: sp.Symbol(name) for name in all_vars}
    symbol_names = {
        var_syms['rh']:         r'\widehat{RH}',
        var_syms['thetae']:     r'\widehat{\theta_e}',
        var_syms['thetaestar']: r'\widehat{\theta_e^*}',
        var_syms['lf']:         r'\mathrm{LF}',
        var_syms['shf']:        r'\mathrm{SHF}',
        var_syms['lhf']:        r'\mathrm{LHF}'}
    ns = {**var_syms,
          'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
          'sqrt':sp.sqrt,'exp':sp.exp,'log':sp.log,'abs':sp.Abs,'cos':sp.cos,'sin':sp.sin}
    try:
        expr = eval(eq_str,{'__builtins__':{}},ns)
        for atom in list(expr.atoms(sp.Float)):
            expr = expr.subs(atom,sp.Float(f'{float(atom):.3g}'))
        return f'${sp.latex(expr,symbol_names=symbol_names)}$'
    except Exception:
        return eq_str

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

with xr.open_dataset(os.path.join(SPLITSDIR,'norm_train.h5'),engine='h5netcdf') as ds:
    firstvar = next(iter(MODELS['nn']['runs'].values()))['fieldvars'][0]
    nsigs    = ds.sizes['sig'] if 'sig' in ds[firstvar].dims else 1

def get_target_nparams(name):
    if name == 'pod_bl':
        with np.load(os.path.join(MODELSDIR,'pod',f'{name}.npz')) as d:
            return int(d['nparams'])
    if name in MODELS['nn']['runs']:
        cfg = MODELS['nn']['runs'][name]
        return get_nn_complexity(cfg['kind'],len(cfg['fieldvars']),nsigs,len(cfg.get('localvars',[])))
    if name in SRCONFIG.get('optimizedeqs',{}):
        return SRCONFIG['optimizedeqs'][name]['refcomplexity']
    return None

# 8 target models — single (time,lat,lon) prediction, no complexity dim
target_results = {}
for name in MODELS_8:
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):
        print(f'Missing predictions: {name}')
        continue
    with xr.open_dataset(filepath) as ds:
        predtp = ds.tp.load()
    if 'seed' in predtp.dims:
        predtp = predtp.mean('seed')
    if 'complexity' in predtp.dims:
        predtp = predtp.isel(complexity=0)
    ytrue,ypred = xr.align(truetp,predtp,join='inner')
    target_results[name] = (ytrue.squeeze(),ypred.squeeze())

# SR Pareto frontier — runs with a complexity dimension (e.g. sr_gauss_all)
pareto_results = {}
for name in SRCONFIG['runs']:
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):
        continue
    with xr.open_dataset(filepath) as ds:
        predtp = ds.tp.load()
    if 'complexity' not in predtp.dims:
        continue
    ytrue,ypred = xr.align(truetp,predtp,join='inner')
    pareto_results[name] = (ytrue.squeeze(),ypred)

print(f'Target models loaded: {len(target_results)}/{len(MODELS_8)}')
print(f'Pareto runs loaded:   {len(pareto_results)}')

In [ ]:
# Compute metrics for target models
target_r2      = {name: get_r2(*target_results[name])     for name in target_results}
target_mse     = {name: get_mse(*target_results[name])    for name in target_results}
target_nparams = {name: get_target_nparams(name)          for name in target_results}

# Build SR Pareto frontier records
frontier_records = []
for run,(ytrue,ypred) in pareto_results.items():
    for c in ypred.complexity.values:
        frontier_records.append(dict(mse=get_mse(ytrue,ypred.sel(complexity=int(c))),nparams=int(c)))
frontier = pareto_front(frontier_records) if frontier_records else []

# R²=0.5 reference line
ref_ytrue  = next(iter(target_results.values()))[0]
mse_r2half = get_mse_at_r2(ref_ytrue,r2_target=0.5)

pplt.rc.update({'figure.dpi':150,'reso':'xx-hi','font.size':8,'label.size':8,'tick.labelsize':8})
fig,axs = pplt.subplots(ncols=2,refwidth=3.6,refheight=3.2,share=False)

# ── Left: R² bar chart ───────────────────────────────────────────────────────
ax   = axs[0]
rows = [(name,target_r2[name]) for name in MODELS_8 if name in target_r2]
rows.sort(key=lambda x: x[1])
labels_sorted = [LABELS_8[n] for n,_ in rows]
r2_sorted     = [r2           for _,r2 in rows]
colors_sorted = [COLORS_8[n]  for n,_ in rows]
ax.barh(labels_sorted,r2_sorted,color=colors_sorted)
for i,(_,r2) in enumerate(rows):
    ax.text(max(r2-0.015,0.01),i,f'{r2:.3f}',va='center',ha='right',fontsize=7,color='w')
ax.format(xlabel=r'$R^2$',grid=False,xminorticks='none',yminorticks='none')

# ── Right: Pareto scatter ─────────────────────────────────────────────────────
ax = axs[1]

# SR Pareto background (all complexity points + frontier line)
if frontier_records:
    ax.scatter([r['mse'] for r in frontier_records],
               [r['nparams'] for r in frontier_records],
               color='#D42028',s=12,alpha=0.35,zorder=2)
    if len(frontier)>1:
        ax.plot([r['mse'] for r in frontier],[r['nparams'] for r in frontier],
                color='#D42028',ls='-',lw=1.2,zorder=3,label='SR Pareto frontier')

# 8 target model points
seen = set()
for name in MODELS_8:
    if name not in target_results or target_nparams.get(name) is None:
        continue
    color = COLORS_8[name]
    label = LABELS_8[name] if name not in seen else None
    ax.scatter(target_mse[name],target_nparams[name],
               color=color,s=60,marker='o',zorder=6,label=label)
    seen.add(name)

# R²=0.5 reference line
ax.axvline(mse_r2half,color='k',ls=':',lw=1)
ax.text(mse_r2half,0.98,r'$R^2>0.5$',rotation=90,va='top',ha='right',
        transform=ax.get_xaxis_transform(),fontsize=7)

ax.format(xlabel=r'MSE (mm$^2$ day$^{-2}$)',ylabel='Model complexity',
          yscale='log',grid=True,yminorticks='none',xminorticks='none')
ax.legend(loc='r',ncols=1,fontsize=7)

pplt.show()
# fig.save('../figs/fig_pareto.jpg')